Remas Rawas · S24109161

 `%pip install pgmpy` 
 

### 1. Build and check the network

In [1]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel"),
])

states = {node: ["yes", "no"] for node in model.nodes()}

heatwave = TabularCPD("heatwave", 2, [[0.30], [0.70]],
                     state_names={"heatwave": states["heatwave"]})
overload = TabularCPD("overload", 2, [[0.60, 0.10], [0.40, 0.90]],
                     evidence=["heatwave"], evidence_card=[2],
                     state_names={node: states[node] for node in ["overload", "heatwave"]})

# Columns for (heatwave, overload): yes/yes, yes/no, no/yes, no/no.
powercut = TabularCPD("powercut", 2, [[0.70, 0.20, 0.50, 0.05],
                                     [0.30, 0.80, 0.50, 0.95]],
                     evidence=["heatwave", "overload"], evidence_card=[2, 2],
                     state_names={node: states[node] for node in ["powercut", "heatwave", "overload"]})
cancel = TabularCPD("cancel", 2, [[0.80, 0.10], [0.20, 0.90]],
                   evidence=["powercut"], evidence_card=[2],
                   state_names={node: states[node] for node in ["cancel", "powercut"]})

model.add_cpds(heatwave, overload, powercut, cancel)
print(model.check_model())

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


True


In [2]:
%pip install pgmpy


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Find the exact probability

In [3]:
result = VariableElimination(model).query(
    variables=["heatwave"], evidence={"cancel": "yes"}, show_progress=False
)
exact = result.get_value(heatwave="yes")
print(f"P(heatwave=yes | cancel=yes) = {exact:.4f}")
print(f"P(heatwave=no  | cancel=yes) = {result.get_value(heatwave='no'):.4f}")

P(heatwave=yes | cancel=yes) = 0.5367
P(heatwave=no  | cancel=yes) = 0.4633


### 3. Estimate it using rejection sampling

In [4]:
samples = BayesianModelSampling(model).rejection_sample(
    evidence=[State("cancel", "yes")], size=10_000, seed=42, show_progress=False
)
yes_count = (samples["heatwave"] == "yes").sum()
estimate = yes_count / len(samples)
print(f"{yes_count:,} out of {len(samples):,} samples had a heatwave.")
print(f"Sampling estimate = {estimate:.4f}")

5,430 out of 10,000 samples had a heatwave.
Sampling estimate = 0.5430


The exact probability was 0.5367, and the rejection sampling estimate from 10,000 samples should be close to it. The estimate can differ slightly because it comes from random samples.
